# 01a linopy

[linopy](https://linopy.readthedocs.io) builds linear programs out of labelled `xarray` arrays.
That makes it the shortest path to a block structure: you name the dimension to split, here
`snapshot`, and every variable and constraint finds its block from its own dimensions. PyPSA
builds its models with linopy, so everything here carries over to [02 PyPSA](02-pypsa.ipynb).

The PIPS-IPM++ backend is not upstream yet, so the workshop installs linopy from a fork.

| sections | run on |
|---|---|
| 1 to 4: build, export, visualize, change the model, reference solve with HiGHS | every machine |
| 5: solve with PIPS-IPM++ | Linux or WSL with the `solver` extra |
| 6: more than one process with MPI | Linux or WSL with the `solver` extra |

In [ ]:
import workshop
import linopy

workshop.check()

## 1. The model

The sample model from [00](00-start-here.ipynb). The data (costs, profiles, the emission budget)
lives in `sample_data.py` next to this notebook, so every framework computes the same numbers.

The cell below writes the model into a file, `linopy_model.py`, instead of running it. That way
this notebook and the parallel run in section 6 use the very same code. The cell after it loads
the file.

Look for the block structure in the code: `cap` has no `snapshot` dimension, `gen` has one. That is
all linopy needs.

In [ ]:
%%writefile linopy_model.py
import pandas as pd
import xarray as xr
from linopy import Model

from sample_data import STORAGE, TECHS, budget, profiles


def build(T=24, techs=TECHS, storage=False):
    snapshots = pd.RangeIndex(T, name="snapshot")
    tech = pd.Index(list(techs), name="tech")
    demand, avail = profiles(T)

    def per_tech(column):
        return xr.DataArray([techs[g][column] for g in tech], coords=[tech])

    invcost, varcost, emission = per_tech(0), per_tech(1), per_tech(2)
    availability = xr.DataArray([avail[techs[g][3]] for g in tech], coords=[tech, snapshots])

    m = Model()
    cap = m.add_variables(lower=0, coords=[tech], name="cap")            # no snapshot: root
    gen = m.add_variables(lower=0, coords=[tech, snapshots], name="gen")
    supply = gen.sum("tech")
    cost = (invcost * cap).sum() + (varcost * gen).sum()

    if storage:
        sto_cap = m.add_variables(lower=0, name="sto_cap")                 # no snapshot: root
        charge = m.add_variables(lower=0, coords=[snapshots], name="charge")
        discharge = m.add_variables(lower=0, coords=[snapshots], name="discharge")
        soc = m.add_variables(lower=0, coords=[snapshots], name="soc")
        # the level carries over from the previous hour; the first hour starts empty
        m.add_constraints(soc - soc.shift(snapshot=1) - STORAGE["efficiency"] * charge
                          + discharge == 0, name="soc")
        m.add_constraints(soc - sto_cap <= 0, name="soc_limit")
        supply = supply + discharge - charge
        cost = cost + STORAGE["investment"] * sto_cap

    m.add_constraints(supply == xr.DataArray(demand, coords=[snapshots]), name="balance")
    m.add_constraints(gen - availability * cap <= 0, name="limit")
    m.add_constraints((emission * gen).sum() <= budget(T), name="carbon")   # linking
    m.add_objective(cost)
    return m

In [ ]:
%run linopy_model.py

m = build()
print(m)

<details><summary>💡 Hint: I changed the model. Why did nothing happen?</summary>

Two cells work together: `%%writefile` saves the code to `linopy_model.py`, and `%run` loads it.
After editing the first, run both again, and then `build()` gives you the new model.
</details>

## 2. Export it and look at the matrix

`PIPSIPMpp.write_parquet` writes the model the way PIPS-IPM++ reads it: the matrix, the bounds and
costs, and the block of every row and column. It does not need the solver or MPI, so it runs on
every machine.

- `n_blocks=4` cuts the `snapshot` dimension into four contiguous slices, one per leaf block.
- `layout="monolithic"` writes one set of files for the whole model. (`"distributed"` writes one
  set per block, see [04](04-deep-dive.ipynb).)
- `names=True` keeps linopy's names for every row and column.

In [ ]:
from pathlib import Path
from linopy.solvers import PIPSIPMpp

Path("data").mkdir(exist_ok=True)


def export(m, name, n_blocks=4):
    return PIPSIPMpp.write_parquet(m, f"data/{name}", layout="monolithic",
                                   n_blocks=n_blocks, names=True)


stem = export(build(), "linopy_base")
print(sorted(p.name for p in Path("data").glob("linopy_base.*")))

`workshop.visualize` runs `pipstools visualize` on the files, shows the picture, and prints the
numbers that decide whether a structure is good:

In [ ]:
workshop.visualize(stem)

How to read it:

- Rows and columns are sorted by block. Each dot is a nonzero.
- The narrow band on the left holds the 2 root columns, `cap[wind]` and `cap[gas]`. Its dots are
  the capacity limits of every hour, which use the capacity of their technology.
- The four coloured squares are the leaf blocks, six hours each: 12 `gen` columns and 18 rows
  (6 balance, 12 limit).
- The grey row at the bottom is the one linking row, the carbon budget. It touches every leaf.

This is the structure from the table in 00, now drawn from the actual matrix.

<details><summary>💡 Hint: does <code>n_blocks</code> count the root?</summary>

No. `n_blocks` is the number of leaves. The root always exists on top, so `n_blocks=4` gives
block `0` (root) plus the leaves `1, 2, 3, 4`.
</details>

<details><summary>💡 Hint: can I run pipstools myself, in a terminal?</summary>

Yes. `workshop.pipstools(...)` prints the command it runs, and `workshop.visualize` is
`pipstools visualize data/linopy_base.amat.parquet --outputfile ...`. Run `pipstools --help` from
the activated environment (`uv run pipstools --help` or `pixi run pipstools --help`).
</details>

## 3. Change the model and watch the matrix

Each exercise changes the model once. Before you run a cell, predict what happens to the three
numbers: root columns, columns per leaf, linking rows. Then compare with the picture.

### Exercise 1: more hours

Two days instead of one, still in 4 blocks, and then in 8.

In [ ]:
workshop.visualize(export(build(T=48), "linopy_T48"))

In [ ]:
workshop.visualize(export(build(T=48), "linopy_T48_8b", n_blocks=8))

The root stays at 2 columns however many hours you add, and there is still one linking row. More
hours make each leaf larger (with 4 blocks) or give more leaves (with 8). This is the property
that makes energy models a good fit: they grow along time, and time goes into the leaves.

Try `T=168`, a week, with 7 blocks of one day each.

### Exercise 2: another technology

`sample_data.py` describes a technology as one row: investment cost, variable cost, emissions and
availability profile. `SOLAR` is a row prepared for you; adding it is one line.

In [ ]:
from sample_data import SOLAR, TECHS

with_solar = TECHS | {"solar": SOLAR}
print(with_solar)
workshop.visualize(export(build(techs=with_solar), "linopy_solar"))

One more root column, `cap[solar]`, and every leaf grows by one `gen` column and one limit row
per hour. The linking row stays alone: the carbon budget now touches solar too, but it was
linking already.

Invent your own technology: a tuple `(investment, variable, emission, profile)` with one of the
profiles `"wind"`, `"solar"` or `"firm"`. A cheap, dirty one such as `(0.2, 0.8, 0.9, "firm")`
changes the answer a lot and the structure not at all.

### Exercise 3: storage

`build(storage=True)` adds the storage from 00. Read the `if storage:` part of the model first.
The line to look at is the level:

```python
soc - soc.shift(snapshot=1) - STORAGE["efficiency"] * charge + discharge == 0
```

`soc.shift(snapshot=1)` is the level of the previous hour. Inside a block that is a local row. In
the first hour of a block it is not. Predict the number of linking rows, then run:

In [ ]:
workshop.visualize(export(build(storage=True), "linopy_storage"))

Four linking rows: the carbon budget, plus one storage level per block boundary. Those three are
2-link rows: each touches only two neighbouring blocks, and you can see them in the picture as
short strokes at the bottom, right at the corners between two blocks. PIPS-IPM++ handles them
much more cheaply than the carbon budget, which touches every block.

The storage capacity `sto_cap` is the third root column.

### Exercise 4: a bad split

`n_blocks` always cuts time into contiguous slices. `Model.blocks` lets you say any split you
like: one integer per snapshot, `1..N`. Here every fourth hour goes to the same block, which is a
perfectly valid split and a very poor one for a model with storage:

In [ ]:
import numpy as np
import xarray as xr

m = build(storage=True)
m.blocks = xr.DataArray(1 + np.arange(24) % 4, dims=["snapshot"])      # round robin
print("block of every hour:", m.blocks.values)

rr = PIPSIPMpp.write_parquet(m, "data/linopy_roundrobin", layout="monolithic", names=True)
workshop.visualize(rr)

Now every storage level links two blocks, and the bottom border is as tall as a leaf. Most of
those rows are still 2-link (block 1 to 2, 2 to 3, 3 to 4). The ones from block 4 back to block 1
are not, because blocks 4 and 1 are not neighbours.

The model is the same and its answer is the same. Only the split changed, and with it how much
work the borders cost.

<details><summary>💡 Hint: then why would anyone use <code>Model.blocks</code>?</summary>

For splits that `n_blocks` cannot express: slices of different lengths, representative days that
are not in order, or a split along another dimension such as regions. A dimension other than
`snapshot` is chosen with `block_dim=`.
</details>

### Exercise 5: your own change

Some ideas, each with a prediction to check against the picture:

- Make the storage cyclic, so the last hour feeds the first: replace `soc.shift(snapshot=1)` by
  `soc.roll(snapshot=1)` in the model file, run both cells of section 1 again, and export
  `build(storage=True)`.
- Give each technology its own emission budget per day instead of one budget for all hours.
- Add a second storage, for example one with a lower efficiency and a lower cost.

<details><summary>💡 Answer: what does the cyclic storage do?</summary>

It adds one linking row that ties the last hour of block 4 to the first hour of block 1. That row
touches two blocks, but not two neighbouring ones, so it is not 2-link: the summary reports 5
linking rows, of which 3 are 2-link. One such row is harmless. A model with many of them is
worth reformulating.
</details>

## 4. The answer, with HiGHS

Before solving with PIPS-IPM++, get a reference answer from HiGHS, which runs everywhere:

In [ ]:
reference = {}
for label, model in (("base", build()), ("storage", build(storage=True))):
    model.solve("highs", output_flag=False)
    reference[label] = model.objective.value
    print(f"{label:8s} objective {model.objective.value:.6f}   capacities",
          model.variables["cap"].solution.round(4).to_series().to_dict())

Storage lowers the cost from about 2.74 to about 2.12: it shifts wind into the hours without
wind, and no gas is built at all.

## 5. Solve with PIPS-IPM++

From here on the cells need the solver: Linux or WSL, with JupyterLab started through
`uv run --extra solver jupyter lab` or `pixi run -e solver lab`. On other machines they are
skipped.

The call is the usual `m.solve`, with the block structure given the same way as for the export.
Anything that is not an interface option, such as `SILENT`, goes to PIPS-IPM++ itself.

In [ ]:
%%requires solver
m = build()
status, condition = m.solve("pipsipmpp", n_blocks=4, block_dim="snapshot", SILENT=True)

In [ ]:
%%requires solver
print(status, condition)
print(f"objective   {m.objective.value:.6f}   (HiGHS {reference['base']:.6f})")
print("capacities ", m.variables["cap"].solution.round(4).to_series().to_dict())
print("carbon dual", round(float(m.constraints["carbon"].dual), 4))

The same answer as HiGHS, to about six figures. PIPS-IPM++ is an interior-point method and stops
within a tolerance of the optimum, while HiGHS ends on an exact vertex, so the last digits may
differ.

The solution lands on the model as with any other linopy solver: `.solution` on the variables,
`.dual` on the constraints.

The solve cell shows what PIPS-IPM++ prints: its parameters, a presolve summary and the residuals
at the end. `SILENT` removes most of the iteration log but not those, which is why the results
get a cell of their own.

<details><summary>💡 Hint: why is the carbon dual negative?</summary>

It is the shadow price of the emission budget: how much the objective changes per extra unit of
budget. Relaxing a `<=` constraint in a minimisation can only make things cheaper, so the value
is negative in this sign convention. Read `-1.48` as "one more unit of emissions saves 1.48".
</details>

### The split changes the work, never the answer

The storage model, solved with the three splits from section 3:

In [ ]:
%%requires solver
splits = {}
for label, blocks in (
    ("contiguous", np.repeat(np.arange(1, 5), 6)),
    ("round robin", 1 + np.arange(24) % 4),
    ("two blocks", np.repeat(np.arange(1, 3), 12)),
):
    m = build(storage=True)
    m.blocks = xr.DataArray(blocks, dims=["snapshot"])
    status, _ = m.solve("pipsipmpp", SILENT=True)
    splits[label] = (status, m.objective.value)

In [ ]:
%%requires solver
for label, (status, objective) in splits.items():
    print(f"{label:12s} {status}  objective {objective:.6f}")
print(f"{'HiGHS':12s} ok  objective {reference['storage']:.6f}")

If changing the split ever changes your objective beyond the solver tolerance, the annotation has
a bug.

### Options

```python
m.solve(
    "pipsipmpp",
    n_blocks=32,
    options_file="tuned.opt",       # a base configuration, one NAME value per line
    LINEAR_LEAF_SOLVER="ma57",      # overrides the file
    SILENT=True,
)
```

The options that linopy consumes itself are `n_blocks`, `block_dim`, `comm`, `write_parquet`,
`parquet_layout`, `parquet_names` and `options_file`. Everything else is forwarded to PIPS-IPM++.

<details><summary>💡 Hint: which leaf solver should I use?</summary>

`mumps` ships with the solver and is the default. The HSL codes `ma27` and `ma57` are often faster,
but have to be obtained separately under the HSL licence; point PIPS-IPM++ at them with the
`HSLLIB` option.
</details>

## 6. More than one process

A notebook is a single process, so everything above solved on one. To use the decomposition, the
same code runs under MPI as a script. The script imports `linopy_model.py` from section 1.

In [ ]:
%%writefile linopy_parallel.py
from mpi4py import MPI
from linopy_model import build

m = build(storage=True)
status, condition = m.solve("pipsipmpp", n_blocks=4, block_dim="snapshot", SILENT=True)
if MPI.COMM_WORLD.Get_rank() == 0:
    print(f"ranks={MPI.COMM_WORLD.Get_size()} {status} {condition} "
          f"objective={m.objective.value:.6f}")

In [ ]:
%%requires solver
for ranks in (1, 2, 4):
    workshop.mpirun("linopy_parallel.py", ranks, show="ranks=")

The same objective on one, two and four processes, within the solver tolerance. The work is
split differently, so the last digits can differ, and so can the number of iterations.

`workshop.mpirun` runs `mpiexec -n 4 python linopy_parallel.py` for you; in a terminal you can type that yourself.

<details><summary>💡 Hint: every process builds the whole model. Is that wasteful?</summary>

For a linopy model, building is cheap compared to solving, and the backend uses the model from
process 0 only. What you must not do is call `solve()` on only some processes: it is collective,
and the others would wait forever.
</details>

<details><summary>💡 Hint: how many processes should I use?</summary>

At most one per leaf block, since PIPS-IPM++ refuses more processes than leaves. Fewer is fine;
leaves then share a process. On a laptop, the number of CPU cores is the useful upper limit.
</details>

## 7. Now your own model

Swap in a linopy model of your own, export it with `export(m, "mine", n_blocks=...)` and look at
it with `workshop.visualize`. Questions to ask:

1. Which of your variables have no block dimension? They end up in the root. Investment
   capacities are fine there. Something per bus and hour without a time dimension is suspicious.
2. How many linking rows are there, and how many are 2-link? Storage and ramping limits give
   2-link rows at the boundaries. Annual limits give one row each.
3. Does a different number of blocks, or a different dimension, give smaller borders?

## 8. Checkpoint

- Why does `cap` end up in the root without you saying so?
- How many linking rows does the storage create with 8 contiguous blocks, and how many with 8
  round-robin blocks?

## Next

[02 PyPSA](02-pypsa.ipynb) if you use PyPSA, which is this notebook with a power system on top.
Otherwise [03 Structure detection](03-structure-detection.ipynb), which finds a structure in a
model that states none.

## Notes

- Linear problems only. A QP or MILP raises `NotImplementedError`.
- Maximisation is handled for you: linopy negates the objective and flips the sign back.
- Warm starts, basis files and log files are ignored with a warning; PIPS-IPM++ has no equivalent.